In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]
tavily_api_key = os.environ["TAVILY_API_KEY"]
print(f"openai_api_key: {openai_api_key}")
print(f"tavily_api_key: {tavily_api_key}")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
chat_model

In [ ]:
def chatbot(state: State):
    return {"messages": [chat_model.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("llm_chatbot", chatbot)

graph_builder.add_edge(START, "llm_chatbot")
graph_builder.add_edge("llm_chatbot", END)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages": [
    HumanMessage("Hi")
]})

In [ ]:
response['messages'][-1].content

In [ ]:
for event in graph.stream({"messages": HumanMessage("Hi how are you")}):
    for value in event.values():
        print(value["messages"][-1].content)

In [ ]:
from langchain_tavily import TavilySearch

talivy_tool = TavilySearch(max_search=2)
talivy_tool.invoke("What is langgraph?")

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a:int, b:int) -> int:
    """Multiply a and b
    Args:
        a(int): first int
        b(int): second int
    
    Returns:
        int: output int
    """
    return a*b

In [ ]:
tools = [talivy_tool, multiply]

In [ ]:
llm_with_tools = chat_model.bind_tools(tools)

In [ ]:
llm_with_tools

In [ ]:
# Stategraph

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm", 
    tools_condition
    )
builder.add_edge("tools", END)

news_graph=builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(news_graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


In [ ]:
# response = news_graph.invoke({"messages": HumanMessage("What is the recent ai news?")})

In [ ]:
response = news_graph.invoke({"messages": HumanMessage("What is 5 multiplied by 2 and then multiply by 10")})
for m in response["messages"]:
    m.pretty_print()

In [ ]:
# ReAct architecture

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


react_builder = StateGraph(State)
react_builder.add_node("tool_calling_llm", tool_calling_llm)
react_builder.add_node("tools", ToolNode(tools))

react_builder.add_edge(START, "tool_calling_llm")
react_builder.add_conditional_edges("tool_calling_llm", tools_condition)
react_builder.add_edge("tools", "tool_calling_llm")

react_graph = react_builder.compile()

from IPython.display import Image, display

try:
    display(Image(react_graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


In [ ]:
response = react_graph.invoke(
    {"messages": HumanMessage("Give me the recent AI news and then tell what is 5*9")}
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()